# Using spectraxai with PyCaret

[PyCaret](https://pycaret.org) is a low-code library that automates training, comparing and evaluating many models. `spectraxai` does not depend on it, but its building blocks are scikit-learn compatible, so they plug straight into PyCaret:

- `Dataset.to_dataframe()` gives PyCaret the table it expects;
- `spectraxai.preprocessing.SpectralPreprocessor` and the selectors of `spectraxai.selection` compose with any regressor into a scikit-learn pipeline, which PyCaret trains and cross-validates like one of its own models (the pre-treatment is refitted inside every fold);
- `spectraxai.models.StandardModel` can be passed to PyCaret like any other estimator.

**Requirements.** This notebook uses the PyCaret 4.0 API (`RegressionExperiment`); at the time of writing only its pre-releases support Python 3.13+, so install it separately, ideally in its own environment:

```
pip install --pre "pycaret>=4.0.0a8"
```

It is therefore not run by the spectraxai test suite. It was last run with PyCaret 4.0.0a8.

## Data

The bundled Greek soil spectral library, predicting soil organic matter (OM) from the reflectance spectra.

In [1]:
from spectraxai.data import load_GR_SSL

dataset = load_GR_SSL()
df = dataset.to_dataframe()
df.shape

(928, 109)

## Set up the experiment

PyCaret holds out 20% of the samples as a test set and cross-validates on the rest. Its own preprocessing is switched off: the spectral pre-treatment is part of each candidate pipeline instead.

In [2]:
from pycaret.regression import RegressionExperiment

experiment = RegressionExperiment(
    target="OM", session_id=0, train_size=0.8, fold=5, preprocess=False
)
experiment.fit(df)
experiment.X_train.shape, experiment.X_test.shape

((742, 108), (186, 108))

## Candidate pipelines

Absorbance followed by a first Savitzky–Golay derivative and SNV, then either a spectraxai PLS model, Ridge regression on the 30 wavelengths most correlated with OM, or a random forest. A random forest on the raw reflectance serves as a baseline.

In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

from spectraxai.models import Model, StandardModel
from spectraxai.preprocessing import SpectralPreprocessor
from spectraxai.selection import SelectKBest
from spectraxai.spectra import SpectralPreprocessing as SP


def pretreatment():
    return SpectralPreprocessor(
        [SP.ABS, (SP.SG1, {"window_length": 7, "polyorder": 2}), SP.SNV],
        wavelengths=dataset.wavelengths,
    )


candidates = {
    "RF (raw reflectance)": RandomForestRegressor(n_estimators=100, random_state=0),
    "ABS+SG1+SNV -> PLS": make_pipeline(
        pretreatment(), StandardModel(Model.PLS, init_hyperparameters={"n_components": 10})
    ),
    "ABS+SG1+SNV -> 30 bands -> Ridge": make_pipeline(
        pretreatment(), SelectKBest(k=30), Ridge()
    ),
    "ABS+SG1+SNV -> RF": make_pipeline(
        pretreatment(), RandomForestRegressor(n_estimators=100, random_state=0)
    ),
}

## Cross-validate every candidate

In [4]:
import pandas as pd

created = {name: experiment.create_model(model) for name, model in candidates.items()}
pd.DataFrame(
    {name: result.metrics.loc["Mean"] for name, result in created.items()}
).T.sort_values("RMSE")

,MAE,MSE,RMSE,R2,RMSLE,MAPE
ABS+SG1+SNV -> PLS,0.3164,0.1913,0.4366,0.5246,0.2081,0.5026
ABS+SG1+SNV -> RF,0.3306,0.2040,0.4505,0.4988,0.2149,0.5493
ABS+SG1+SNV -> 30 bands -> Ridge,0.3482,0.2253,0.4738,0.4446,0.2274,0.5777
RF (raw reflectance),0.3886,0.2755,0.5234,0.3305,0.2521,0.7016


`compare_models` accepts the same estimators and returns the best one, which can then be scored on the held-out test set.

In [5]:
comparison = experiment.compare_models(include=list(candidates.values()), sort="RMSE")
experiment.predict_model(comparison.best).metrics

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Pipeline,0.3266,0.1872,0.4327,0.4243,0.2233,0.5303
